In [7]:
import json
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import nltk
from nltk.stem import SnowballStemmer
import random

print('Good imports!')


Good imports!


In [6]:
nltk.download('punkt')
nltk.download('punkt_tab')
stemmer = SnowballStemmer('russian')

# 1. Загрузка данных
with open('intents.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

words = []
labels = []
docs_x = []
docs_y = []

for intent in data['intents']:
    for pattern in intent['patterns']:
        wrds = nltk.word_tokenize(pattern.lower())
        words.extend(wrds)
        docs_x.append(wrds)
        docs_y.append(intent['tag'])
        
    if intent['tag'] not in labels:
        labels.append(intent['tag'])

words = [stemmer.stem(w) for w in words if w != '?']
words = sorted(list(set(words)))
labels = sorted(labels)

training = []
output = []
out_empty = [0] * len(labels)

for x, doc in enumerate(docs_x):
    bag = []
    wrds = [stemmer.stem(w) for w in doc]
    
    for w in words:
        bag.append(1) if w in wrds else bag.append(0)
        
    output_row = list(out_empty)
    output_row[labels.index(docs_y[x])] = 1
    
    training.append(bag)
    output.append(output_row)

X_train = np.array(training)
y_train = np.array(output)

model = Sequential([
    Dense(128, input_shape=(len(X_train[0]),), activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(len(y_train[0]), activation='softmax') # Softmax для классификации
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=200, batch_size=5, verbose=1)

print("Модель успешно обучена!")


[nltk_data] Downloading package punkt to /home/kota/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/kota/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/usr/lib/python3.14/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.1765 - loss: 1.1537
Epoch 2/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4706 - loss: 1.0968
Epoch 3/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4118 - loss: 1.0871
Epoch 4/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2353 - loss: 1.1274    
Epoch 5/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2353 - loss: 1.1223    
Epoch 6/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4118 - loss: 1.0824
Epoch 7/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4118 - loss: 1.0243
Epoch 8/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4118 - loss: 1.0743
Epoch 9/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5294 - loss: 1.0601
Epoch 10/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5882 - loss: 1.0105
Epoch 11/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6471 - loss: 0.9806
Epoch 12/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.5

In [9]:
def bag_of_words(s, words):
    sentence_words = nltk.word_tokenize(s.lower())
    sentence_words = [stemmer.stem(w) for w in sentence_words]
    
    bag = [0] * len(words)
    for se in sentence_words:
        for i, w in enumerate(words):
            if w == se:
                bag[i] = 1
                
    return np.array([bag])

def chat():
    print("Бот запущен! Напиши 'выход', чтобы остановить общение.\n" + "-"*50)
    
    while True:
        inp = input("Вы: ")
        if inp.lower() == "выход":
            print("Бот: До встречи!")
            break
        bow = bag_of_words(inp, words)
        
        results = model.predict(bow, verbose=0)[0]
        
        results_index = np.argmax(results)
        tag = labels[results_index]
        
        if results[results_index] > 0.7:
            for tg in data["intents"]:
                if tg['tag'] == tag:
                    responses = tg['responses']
            
            print(f"Бот: {random.choice(responses)}")
        else:
            print("Бот: Извини, я не совсем понял. Перефразируй, пожалуйста.")
        print("-"*50)

chat()


Бот запущен! Напиши 'выход', чтобы остановить общение.
--------------------------------------------------


Бот: Приветствую!
--------------------------------------------------
Бот: Извини, я не совсем понял. Перефразируй, пожалуйста.
--------------------------------------------------
Бот: Извини, я не совсем понял. Перефразируй, пожалуйста.
--------------------------------------------------
Бот: Рад тебя видеть! Какой у тебя вопрос?
--------------------------------------------------
Бот: Будут вопросы — пиши.
--------------------------------------------------
Бот: До встречи!
